# Question 2 — Who are the loopers in practice?

**Question:** Who are the loopers in practice — retail, leveraged vaults, the issuer's own market makers? How would you verify this on-chain, and how does the answer change the DAO's risk?

**Method, in one line:** find every address that currently holds *both* sUSDe collateral (aToken) and USDe debt (variable debt token) on this Pool, then profile them. Pinned to block 25,682,519 for reproducibility.

## 2.1 — Identifying loopers on-chain

An address only shows up in Aave's data as holding sUSDe collateral *and* USDe debt simultaneously if it's running the loop — sUSDe's standalone LTV is 0% (Section 0), so sUSDe can never back a USDe borrow outside the E-mode categories this loop uses. The intersection of "nonzero sUSDe aToken balance" and "nonzero USDe variable-debt-token balance" is therefore, by construction, the looper set. (Heads up this takes a long time to compute)

**Steps:**
1. Pull every `Transfer` event ever emitted by the sUSDe aToken (`0x4579a27a...`) and the USDe variable debt token (`0x015396E1...`) — this gives the complete set of addresses that have *ever* held either token (9,071 unique addresses; anyone with a current nonzero balance must have appeared in at least one Transfer, so this set is provably complete).
2. Batch-query current balances for all 9,071 candidates on both tokens via Multicall3, at the pinned block.
3. Keep only addresses with both balances > 0.

In [ ]:
from web3 import Web3
import json, requests, time

RPC = "https://eth.drpc.org"
w3 = Web3(Web3.HTTPProvider(RPC))
BLOCK_NUMBER = 25682519

SUSDE_ATOKEN = w3.to_checksum_address("0x4579a27aF00A62C0EB156349f31B345c08386419")
USDE_VTOKEN  = w3.to_checksum_address("0x015396E1F286289aE23a762088E863b3ec465145")
MULTICALL3   = w3.to_checksum_address("0xcA11bde05977b3631167028862bE2a173976CA11")
TRANSFER_TOPIC = w3.keccak(text="Transfer(address,address,uint256)").hex()

# Step 1: paginate eth_getLogs in <=10k-block windows -- rpc.mevblocker.io enforces this range
# cap directly (error -32602, "range X exceeds limit of 10000"), so chunk to it up front rather
# than wait for the error. If a chunk *still* errors (e.g. an unusually log-dense window hits a
# provider-side result-count cap too), bisect that chunk further as a fallback.
def get_logs(address, from_block, to_block, chunk_size=9_999, max_retries=5):
    results = []
    start = from_block
    while start <= to_block:
        end = min(start + chunk_size, to_block)
        payload = {"jsonrpc": "2.0", "method": "eth_getLogs", "id": 1,
                   "params": [{"address": address, "topics": [TRANSFER_TOPIC],
                                "fromBlock": hex(start), "toBlock": hex(end)}]}
        for attempt in range(max_retries):
            try:
                data = requests.post("https://rpc.mevblocker.io", json=payload, timeout=30).json()
                break
            except requests.exceptions.RequestException:
                if attempt == max_retries - 1:
                    raise
                time.sleep(2 ** attempt)
        if "error" in data:
            if end > start:
                mid = (start + end) // 2
                results += get_logs(address, start, mid, chunk_size, max_retries)
                results += get_logs(address, mid + 1, end, chunk_size, max_retries)
            else:
                raise RuntimeError(data["error"])
        else:
            results.extend(data["result"])
        start = end + 1
    return results

def addr_from_topic(t):
    return Web3.to_checksum_address("0x" + t[-40:])

candidates = set()
for token in [SUSDE_ATOKEN, USDE_VTOKEN]:
    logs = get_logs(token, 0, BLOCK_NUMBER)
    for log in logs:
        candidates.add(addr_from_topic(log["topics"][1]))
        candidates.add(addr_from_topic(log["topics"][2]))
candidates.discard(Web3.to_checksum_address("0x" + "0"*40))
candidates = sorted(candidates)  # all checksummed, consistent with balance dict keys below
print(f"Unique addresses that ever held either token: {len(candidates)}")

In [ ]:
import time

MULTICALL_ABI = json.loads('''[
 {"name":"aggregate3","type":"function","stateMutability":"view",
  "inputs":[{"name":"calls","type":"tuple[]","components":[
    {"name":"target","type":"address"},{"name":"allowFailure","type":"bool"},{"name":"callData","type":"bytes"}]}],
  "outputs":[{"name":"returnData","type":"tuple[]","components":[
    {"name":"success","type":"bool"},{"name":"returnData","type":"bytes"}]}]}
]''')
multicall = w3.eth.contract(address=MULTICALL3, abi=MULTICALL_ABI)

def with_retry(fn, retries=5, base_delay=2):
    """Retry with exponential backoff -- public RPC endpoints occasionally 429 under sustained multicall traffic."""
    for attempt in range(retries):
        try:
            return fn()
        except Exception as e:
            if attempt == retries - 1:
                raise
            time.sleep(base_delay * (2 ** attempt))

def balanceof_calldata(addr):
    return bytes.fromhex("70a08231") + bytes.fromhex(addr[2:].rjust(64, "0"))

def batch_balances(token, addr_list, chunk=400):
    results = {}
    for i in range(0, len(addr_list), chunk):
        batch = [w3.to_checksum_address(a) for a in addr_list[i:i+chunk]]
        calls = [(token, True, balanceof_calldata(a)) for a in batch]
        out = with_retry(lambda: multicall.functions.aggregate3(calls).call(block_identifier=BLOCK_NUMBER))
        for a, (success, data) in zip(batch, out):
            results[a] = int.from_bytes(data, "big") if success and len(data) == 32 else 0
    return results

WAD = 10**18
susde_bal = batch_balances(SUSDE_ATOKEN, candidates)
usde_debt_bal = batch_balances(USDE_VTOKEN, candidates)

total_susde = sum(susde_bal.values()) / WAD
total_debt = sum(usde_debt_bal.values()) / WAD
print(f"Total sUSDe collateral on this Pool: {total_susde:,.0f}")
print(f"Total USDe variable debt on this Pool: {total_debt:,.0f}  (matches Section 0's reserve read almost exactly)")

loopers = {a: {"collateral": susde_bal[a]/WAD, "debt": usde_debt_bal[a]/WAD}
           for a in candidates if susde_bal[a] > 0 and usde_debt_bal[a] > 0}
print(f"\nLoopers (both legs nonzero): {len(loopers)} addresses")

## 2.2 — How big is the loop, relative to the whole reserve?

In [ ]:
total_looper_collateral = sum(v["collateral"] for v in loopers.values())
total_looper_debt = sum(v["debt"] for v in loopers.values())

print(f"Loopers hold {total_looper_collateral:,.0f} of {total_susde:,.0f} sUSDe collateral = {total_looper_collateral/total_susde:.1%} of the entire reserve")
print(f"Loopers hold {total_looper_debt:,.0f} of {total_debt:,.0f} USDe debt          = {total_looper_debt/total_debt:.1%} of the entire reserve")

ranked = sorted(loopers.items(), key=lambda x: -x[1]["debt"])
top10 = sum(v["debt"] for _, v in ranked[:10])
top3 = sum(v["debt"] for _, v in ranked[:3])
print(f"\nWithin the looper set itself:")
print(f"  Top 10 addresses (of {len(loopers)}) = {top10/total_looper_debt:.1%} of all looper debt")
print(f"  Top 3 addresses                = {top3/total_looper_debt:.1%} of all looper debt")

small = sum(v["debt"] for _, v in ranked if v["debt"] < 10_000)
large = sum(v["debt"] for _, v in ranked if v["debt"] >= 1_000_000)
n_small = sum(1 for _, v in ranked if v["debt"] < 10_000)
n_large = sum(1 for _, v in ranked if v["debt"] >= 1_000_000)
print(f"\nRetail-scale positions (<$10k debt):  {n_small} addresses, {small/total_looper_debt:.3%} of looped value")
print(f"Large positions (>=$1M debt):          {n_large} addresses, {large/total_looper_debt:.1%} of looped value")

Loopers hold 279,863,276 of 357,087,043 sUSDe collateral = 78.4% of the entire reserve
Loopers hold 295,154,950 of 582,621,785 USDe debt          = 50.7% of the entire reserve

Within the looper set itself:
  Top 10 addresses (of 100) = 85.0% of all looper debt
  Top 3 addresses                = 57.3% of all looper debt

Retail-scale positions (<$10k debt):  51 addresses, 0.015% of looped value
Large positions (>=$1M debt):          26 addresses, 96.6% of looped value


Loopers account for roughly **half of all USDe borrowing** and the large majority of sUSDe collateral use. Within the loop it's a barbell: ~half the addresses are retail-scale (<$10k, 0.015% of looped value combined); the footprint is set by ~26 large addresses.

## 2.3 — How much leverage are they actually running?

**Correction:** an earlier version priced collateral by hand (raw balance × queried oracle price), which produced absurd leverage/HF outliers. Cross-checking against Aave's own `getUserAccountData()` — the Pool's authoritative collateral/debt/HF for an account — showed the manual price source didn't match Aave's oracle, and some addresses carry debt beyond just USDe. Fixed by calling `getUserAccountData()` directly for every looper; leverage is $L = C/(C-D)$.

In [ ]:
POOL_ACCOUNT_ABI = json.loads('''[
 {"name":"getUserAccountData","type":"function","stateMutability":"view",
  "inputs":[{"name":"user","type":"address"}],
  "outputs":[{"name":"totalCollateralBase","type":"uint256"},{"name":"totalDebtBase","type":"uint256"},
    {"name":"availableBorrowsBase","type":"uint256"},{"name":"currentLiquidationThreshold","type":"uint256"},
    {"name":"ltv","type":"uint256"},{"name":"healthFactor","type":"uint256"}]}
]''')
pool_account = w3.eth.contract(address=w3.to_checksum_address("0x87870Bca3F3fD6335C3F4ce8392D69350B4fA4E2"), abi=POOL_ACCOUNT_ABI)

def account_data_calldata(addr):
    return pool_account.encodeABI(fn_name="getUserAccountData", args=[addr])

BASE = 1e8   # Aave "Base currency" units (8 decimals, ~USD)
HF_UNIT = 1e18

lev_rows = []
addr_list = list(loopers.keys())
for i in range(0, len(addr_list), 100):
    batch = addr_list[i:i+100]
    calls = [(pool_account.address, True, account_data_calldata(a)) for a in batch]
    out = with_retry(lambda: multicall.functions.aggregate3(calls).call(block_identifier=BLOCK_NUMBER))
    for a, (success, data) in zip(batch, out):
        if not success or len(data) < 192:
            continue
        collateral_base, debt_base, avail, lt, ltv, hf = w3.codec.decode(
            ["uint256","uint256","uint256","uint256","uint256","uint256"], data)
        c, d = collateral_base / BASE, debt_base / BASE
        equity = c - d
        leverage = c / equity if equity > 0 else float("inf")
        lev_rows.append((a, c, d, equity, leverage, hf / HF_UNIT, ltv/10000, lt/10000))

lev_rows.sort(key=lambda x: -x[2])
lev_by_addr = {r[0]: r for r in lev_rows}

# Dust (<$10k debt) distorts simple averages -- report separately.
dust = [r for r in lev_rows if r[2] < 10_000]
near_liq = [r for r in lev_rows if r[2] >= 10_000 and r[5] < 1.0]
normal = [r for r in lev_rows if r[2] >= 10_000 and r[5] >= 1.0]

print(f"n={len(lev_rows)}: {len(dust)} dust (<$10k debt), {len(near_liq)} near-liquidation (HF<1), {len(normal)} normal")
if near_liq:
    print("\nNear-liquidation, non-dust:")
    for a, c, d, e, lev, hf, ltv, lt in near_liq:
        print(f"  {a}  collateral=${c:,.0f}  debt=${d:,.0f}  HF={hf:.4f}  leverage={lev:.2f}x")
else:
    print("(None -- the earlier flagged near-liquidation positions were an artifact of the price-reconstruction bug, not real.)")

import statistics
levs = [r[4] for r in normal]
weighted = sum(r[4]*r[2] for r in normal) / sum(r[2] for r in normal)
print(f"\nAmong the {len(normal)} normal-range, non-dust loopers:")
print(f"  Median leverage:   {statistics.median(levs):.2f}x")
print(f"  Debt-weighted avg: {weighted:.2f}x")
print(f"  Max observed:      {max(levs):.2f}x")

above_92pct_origination = [r for r in normal if r[4] > 12.5]
print(f"\nPositions exceeding the 12.5x origination-time cap (still healthy, HF>=1): {len(above_92pct_origination)}")
for a, c, d, e, lev, hf, ltv, lt in above_92pct_origination[:5]:
    print(f"  {a}  leverage={lev:.2f}x  HF={hf:.4f}  current LTV in use={d/c:.2%}  LT={lt:.2%}")

buckets = {"<5x":0, "5-8x":0, "8-10x":0, "10-12.5x":0, "12.5-16.67x (above origination cap, below LT ceiling)":0}
for lev in levs:
    if lev < 5: buckets["<5x"] += 1
    elif lev < 8: buckets["5-8x"] += 1
    elif lev < 10: buckets["8-10x"] += 1
    elif lev < 12.5: buckets["10-12.5x"] += 1
    else: buckets["12.5-16.67x (above origination cap, below LT ceiling)"] += 1
print("\nDistribution:")
for k, n in buckets.items():
    print(f"  {k:55s}: {n}")

n=100: 49 dust (<$10k debt), 0 near-liquidation (HF<1), 51 normal
(None -- the earlier flagged near-liquidation positions were an artifact of the price-reconstruction bug, not real.)

Among the 51 normal-range, non-dust loopers:
  Median leverage:   11.44x
  Debt-weighted avg: 10.99x
  Max observed:      14.42x

Positions exceeding the 12.5x origination-time cap (still healthy, HF>=1): 5
  0xd7583E3CF08bbcaB66F1242195227bBf9F865Fda  leverage=12.74x  HF=1.0201  current LTV in use=92.15%  LT=94.00%
  0xd6c757043e7d59088969B188923C62fa960aFE9B  leverage=12.72x  HF=1.0202  current LTV in use=92.14%  LT=94.00%
  0x9Eb891dA0e35488346d39D41beD182c03B842595  leverage=13.86x  HF=1.0131  current LTV in use=92.78%  LT=94.00%
  0x1cb432f80416f07db4420B8c20447924A0658894  leverage=14.39x  HF=1.0102  current LTV in use=93.05%  LT=94.00%
  0xc6dD9976066F3364b4D6A72cD4F1fA0468327Aa7  leverage=14.42x  HF=1.0100  current LTV in use=93.07%  LT=94.00%

Distribution:
  <5x                                  

**Corrected reading:** zero loopers are currently below HF 1.0 — the earlier near-liquidation results were a methodology artifact, now retracted. Median non-dust leverage is **~11–12x**; a few positions sit above 12.5x while still healthy. That's expected, not a bug: 12.5x is the *origination*-LTV (92%) ceiling, but liquidation only happens at the 94% LT, i.e. up to 16.67x. A position opened at the cap drifts upward as interest accrues and stays healthy the whole way, up to 94%.

## 2.4 — Profiling the top loopers: retail, vaults, or issuer-linked?

Three checks on the top 10 addresses by debt size:

1. **EOA vs. contract** (`eth_getCode`) — vault protocols (DeFi Saver, Instadapp, Gearbox) operate through per-user proxy contracts, so vault-driven activity should show up as contracts, not wallets.
2. **Behavioral signals** — tx count, idle balance, multi-chain presence, Etherscan name tag.
3. **Direct USDe mint history** — only KYC'd institutional counterparties mint directly from Ethena's `EthenaMinting` contract (`Transfer` with `from=0x0`). Mint history is on-chain proof of being an Ethena-approved counterparty, not a retail secondary buyer.

In [ ]:
top10 = ranked[:10]

USDe = w3.to_checksum_address("0x4c9EDD5852cd905f086C759E8383e09bff1E68B3")
ZERO_TOPIC = "0x" + "0"*64

print(f"{'address':<44}{'type':<10}{'debt ($)':>14}{'leverage':>10}{'HF':>8}  {'mint_events':>12}")
for addr, v in top10:
    code_ = with_retry(lambda: w3.eth.get_code(w3.to_checksum_address(addr), block_identifier=BLOCK_NUMBER))
    kind = "EOA" if len(code_) == 0 else "CONTRACT"
    lev_row = lev_by_addr[addr]
    lev, hf = lev_row[4], lev_row[5]
    to_topic = "0x" + "0"*24 + addr[2:].lower()
    payload = {"jsonrpc":"2.0","method":"eth_getLogs","id":1,
        "params":[{"address":USDe,"topics":[TRANSFER_TOPIC, ZERO_TOPIC, to_topic],
                   "fromBlock":"0x0","toBlock":hex(BLOCK_NUMBER)}]}
    mints = len(with_retry(lambda: requests.post("https://rpc.mevblocker.io", json=payload, timeout=20).json()).get("result", []))
    # NB: leverage/HF are whole-account figures from getUserAccountData (2.3), which can
    # differ slightly from a sUSDe/USDe-only view for addresses with other positions.
    print(f"{addr:<44}{kind:<10}{v['debt']:>14,.0f}{lev:>9.2f}x{hf:>8.3f}  {mints:>12d}")

address                                     type            debt ($)  leverage      HF   mint_events
0xCf0a12CBd8088fc5f84ad431E71787157041cD69  EOA           74,068,359     9.95x   1.023             0
0x6142EB927529974c5cDEd66dafc57cB5AaaF73Ab  EOA           50,084,699    11.73x   1.028             0
0x42715bA91deDa3c692B9F540CEe2FBb4daE78bBB  EOA           45,096,733    12.49x   1.022             0
0x086eb9C2B14dD657ae77D07dF9115A2F946CE327  EOA           34,322,313    12.42x   1.022             0
0xd7583E3CF08bbcaB66F1242195227bBf9F865Fda  EOA           14,200,347    12.74x   1.020             0
0x6a73204dB71F8e054bf9A0680b02Ae96f700b595  EOA           10,727,582     8.31x   1.046             0
0x8933850c117bAf7B6423fBB996b4EfA23B67f64a  EOA            7,959,707    12.19x   1.024             0
0x17b6AA44f45a732487F95b524f15053e8149376F  EOA            5,323,044     2.60x   1.528             0
0xd6c757043e7d59088969B188923C62fa960aFE9B  EOA            4,931,008    12.72x   1.020     

**Results:** all top 10 are EOAs, not vault-protocol contracts. Leverage varies 2.6x–12.9x — not one uniform strategy. Manual Etherscan check on the top 4: minimal idle balance (\$47–\$29k against \$34–74M positions), high tx counts (625–2,121), multi-chain, **no public name tag**.

Four of the top 10 have direct USDe mint history from Ethena's minting contract (one, 43 times) — on-chain evidence that at least part of this capital is Ethena-approved institutional counterparties, not retail or third-party vaults.

## 2.5 — Wallet clustering: attribution to Yuzu Money / Ouroboros Capital

**Source:** Arkham Intelligence entity-clustering, run independently on the top 10 wallets from 2.4 — not this notebook's own methodology, and not independently reproducible. Per that analysis, the top 10 cluster to one labeled entity, **Yuzu Money / Ouroboros Capital**, with heavy outflows between wallets in the cluster.

**Independently checked:**
- Ouroboros Capital is a DeFi yield fund; Yuzu Money is its yield-bearing stablecoin (yzUSD/syzUSD, ERC-4626). Pooled depositor capital is redeployed into yield strategies (Euler, Pendle publicly disclosed; Aave not named, but structurally consistent) — per their own [docs/posts](https://x.com/OuroborosCap8/status/1983482990119571487).
- They've disclosed routing capital through a Fordefi MPC wallet restricted to whitelisted interactions — MPC wallets present as EOAs, consistent with 2.4's finding.
- Checked for direct USDe/sUSDe transfers between the top 10 themselves: none found. Inconclusive — the cluster's other associated addresses (e.g. a treasury) aren't identified here.

**If this holds:** the loop's dominant participant is most likely **one external fund** redeploying its own depositors' capital, not a diversified set of counterparties (risk implications in 2.7).

## 2.6 — Verdict

- **Not retail**: 51 of 100 loopers hold <\$10k each, but together that's 0.015% of looped value.
- **Not generic leveraged-vault protocols**: largest positions are EOA-held, not routed through DeFi Saver/Instadapp/Gearbox-style proxies.
- **Most likely one external DeFi yield fund**: per Arkham clustering (2.5, externally sourced, unverified by this notebook), the top 10 attribute to Yuzu Money / Ouroboros Capital, whose disclosed operating model (MPC wallets, whitelisted interactions) is consistent with the EOA pattern found on-chain.
- **Also partly issuer-linked**: several top addresses transact directly with Ethena's minting contract — whitelisted institutional counterparties, not mutually exclusive with the Ouroboros attribution.
- **Highly concentrated**: top 3 = 57% of looped debt; if Arkham's attribution holds, that's largely one entity.
- **High but healthy leverage**: median non-dust ~11–12x; some positions exceed 12.5x while HF > 1, having aged toward the 94% LT (up to ~16.67x). Zero loopers currently below HF 1.0.

## 2.7 — How this changes the DAO's risk

1. **Roughly half the USDe market.** Loopers hold 50.7% of USDe debt — utilization, rate dynamics, and a large share of DAO revenue on this reserve are tied to this one arbitrage persisting.
2. **Concentration is likely single-entity, not diversified.** 57% of looped debt sits in 3 addresses, and Arkham's clustering suggests the top 10 are one fund — a single operational decision, not a market-wide shift, could trigger a large simultaneous deleveraging.
3. **An external risk layer: the fund's own depositors.** If Ouroboros is redeploying pooled depositor capital, Aave's exposure is coupled to Yuzu's own redemption dynamics — a run on Yuzu could force a rapid unwind entirely outside the DAO's visibility.
4. **Issuer-linked capital isn't risk-neutral.** Several top addresses also mint directly from Ethena — the DAO can't assume purely profit-maximizing responses to parameter changes from this segment.
5. **Opacity compounds all of this.** No public Etherscan label on any top address; the Ouroboros/Yuzu attribution itself needs independent verification before it drives a risk decision.
6. **The liquidation cushion is smaller than it looks.** Median leverage (11–12x) sits below the 16.67x liquidation ceiling and no looper is unhealthy today, but leverage drifts upward from the 92% origination cap toward the 94% LT as interest accrues — that cushion compresses by default over time.

## 2.8 — Cross-check: top loopers via Dune

**Source:** [dune.com/kevinlcai/usde-largest-loopers](https://dune.com/kevinlcai/usde-largest-loopers), a separately-built Dune query — live, not pinned to block 25,682,519. Adds an explicit `is_loop_position` classification and each address's share of *total* reserve debt.

| rank | address | type | is_loop | USDe debt | collateral (USDe-equiv) | LTV | implied leverage | share of loop debt | share of total debt | # borrows | first borrow | last borrow |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| 1 | `0xcf0a12cb...41cd69` | EOA | ✓ | $73,866,079 | $83,983,775 | 87.95% | 8.30x | 32.97% | 23.44% | 38 | 2026-02-06 | 2026-08-04 |
| 2 | `0x6142eb92...5aaaf73ab` | EOA | ✗ | $49,139,918 | $112,453,261 | 43.70% | 1.78x | 15.59% | — | 43 | 2025-10-01 | 2026-02-02 |
| 3 | `0x42715ba9...dae78bbb` | EOA | ✓ | $44,909,021 | $49,023,101 | 91.61% | 11.92x | 20.04% | 14.25% | 54 | 2026-02-20 | 2026-08-01 |
| 4 | `0x086eb9c2...946ce327` | EOA | ✓ | $34,298,854 | $37,328,673 | 91.88% | 12.32x | 15.31% | 10.88% | 89 | 2026-07-01 | 2026-08-04 |
| 5 | `0x901431e0...97d3be71` | EOA | ✗ | $22,953,232 | $482,737 | 4754.81% | −0.02x | 7.28% | — | 86 | 2025-08-10 | 2026-07-21 |
| 6 | `0xd7583e3c...9f865fda` | EOA | ✓ | $14,178,045 | $15,410,893 | 92.00% | 12.50x | 6.33% | 4.50% | 6 | 2026-07-16 | 2026-07-20 |
| 7 | `0x6a73204d...96f700b595` | EOA | ✓ | $10,678,166 | $12,196,577 | 87.55% | 8.03x | 4.77% | 3.39% | 12 | 2026-03-01 | 2026-07-21 |
| 8 | `0x8933850c...23b67f64a` | EOA | ✓ | $7,950,925 | $8,671,152 | 91.69% | 12.04x | 3.55% | 2.52% | 7 | 2026-07-03 | 2026-07-23 |
| 9 | `0x17b6aa44...8149376f` | EOA | ✗ | $5,320,000 | $8,652,902 | 61.48% | 2.60x | 1.69% | — | 3 | 2026-07-28 | 2026-07-30 |
| 10 | `0xd6c75704...c62fa960afe9b` | EOA | ✓ | $4,916,565 | $5,352,163 | 91.86% | 12.29x | 2.19% | 1.56% | 6 | 2026-07-16 | 2026-07-16 |
| 11 | `0xea177673...f5b60add4` | contract | ✓ | $4,369,052 | $4,895,275 | 89.25% | 9.30x | 1.95% | 1.39% | 20 | 2026-07-24 | 2026-08-04 |
| 12 | `0x256c7584...39eb298cd` | EOA | ✗ | $4,067,960 | $4,192,224 | 97.04% | 33.74x | 1.29% | — | 60 | 2026-01-30 | 2026-07-03 |
| 13 | `0x05d487ca...6297353a` | contract | ✓ | $3,928,059 | $4,342,663 | 90.45% | 10.47x | 1.75% | 1.25% | 11 | 2026-03-19 | 2026-07-27 |
| 14 | `0x9eb891da...c03b842595` | EOA | ✓ | $3,538,859 | $3,828,136 | 92.44% | 13.23x | 1.58% | 1.12% | 152 | 2025-05-16 | 2026-08-03 |
| 15 | `0x0370046d...f561f6b` | EOA | ✓ | $2,877,830 | $3,138,486 | 91.69% | 12.04x | 1.28% | 0.91% | 10 | 2026-07-06 | 2026-07-09 |
| 16 | `0xd2259a5f...edd04c2b` | EOA | ✗ | $2,386,000 | $2,840,292 | 84.01% | 6.25x | 0.76% | — | 11 | 2026-06-09 | 2026-06-25 |
| 17 | `0x99d6d730...99314c301` | contract | ✓ | $1,943,067 | $2,136,260 | 90.96% | 11.06x | 0.87% | 0.62% | 33 | 2025-09-26 | 2026-07-27 |
| 18 | `0xc3c14cdd...0f530b6786` | EOA | ✓ | $1,888,059 | $2,062,268 | 91.55% | 11.84x | 0.84% | 0.60% | 10 | 2026-07-06 | 2026-07-09 |
| 19 | `0x6c79f721...bd505ab32c` | contract | ✓ | $1,730,550 | $1,893,917 | 91.37% | 11.59x | 0.77% | 0.55% | 11 | 2026-07-09 | 2026-07-21 |
| 20 | `0x8661f478...c9cf20c6` | EOA | ✓ | $1,464,688 | $1,600,571 | 91.51% | 11.78x | 0.65% | 0.46% | 27 | 2026-04-02 | 2026-07-24 |

**Discrepancy:** address `0x256c7584...` (row 12) shows LTV 97.04%/33.74x here vs. ~92%/12.28x from Section 2.3's on-chain `getUserAccountData()` at the same block. The formula matches; the difference is collateral valuation — Dune's figure is ~5.6% below Aave's own `totalCollateralBase`, consistent with pricing sUSDe off the raw ERC-4626 rate rather than Aave's oracle (the same bug Section 2.3 fixed). **For anything liquidation-relevant, `getUserAccountData()` is authoritative**; this table is useful for breadth, not as a leverage/LTV source.

**What `is_loop_position` adds:** 5 of the top 20 by raw debt aren't genuine loop positions. The significant reclassification is rank 2 — \$49.1M debt but only 43.70% LTV against \$112.5M collateral, a large sUSDe depositor who also borrows, not a tight loop. Rows 5, 9, 16 are smaller versions of the same pattern. The ~50% loop-share topline (2.2) holds, but true concentration among *intentional* loops is somewhat tighter than the raw ranking implies.